# Demo 04b — Naive Bayes for Classification: Discrete and Continuous Features

**Class 04 · Block 2** (slides: `slides/04_probabilistic_models.md`)

Five live demos on the **spam** data (Spambase: 4601 e-mails, 57 features, 39% spam):

* **N1 — Naive Bayes by hand and by counting:** a six-e-mail corpus, Laplace smoothing, then Bernoulli NB (discrete
  features) on the spam data.
* **N2 — Gaussian Naive Bayes** (continuous features), and discretizing instead (categorical features).
* **N3 — Naive Bayes versus logistic regression:** generative or discriminative, data size, and probabilities.
* **N4 — Calibration:** are the probabilities right? Log-loss, Brier score, reliability, Platt and isotonic scaling.
* **N5 — Decisions with costs:** the Bayes-optimal threshold.

In [ ]:
import warnings
from pathlib import Path

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, confusion_matrix, log_loss, matthews_corrcoef
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB, CategoricalNB, GaussianNB
from sklearn.preprocessing import KBinsDiscretizer, StandardScaler

plt.rcParams["figure.figsize"] = (9, 4)
DATA = Path("../../datasets")

## N1 — Naive Bayes

**Bayes' rule** turns "how likely is this e-mail if it is spam?" into "how likely is spam, given this e-mail?":

$$P(c \mid x) = \frac{P(c)\,P(x \mid c)}{P(x)} \;\propto\; P(c)\,P(x \mid c)$$

$P(x \mid c)$ over 57 features would need a table with $2^{57}$ entries. The **naive** assumption: given the class, the
features are independent,

$$P(x \mid c) = \prod_j P(x_j \mid c) \quad\Longrightarrow\quad \hat c = \arg\max_c\; \Big[\log P(c) + \sum_j \log P(x_j \mid c)\Big]$$

Each factor is estimated **by counting** (maximum likelihood): there is no gradient and no iteration. We work with logs:
a product of 57 small probabilities underflows, a sum does not.

### A corpus small enough to compute by hand

Three words (`free`, `money`, `meeting`); 1 means "the e-mail contains the word".

In [ ]:
words = ["free", "money", "meeting"]
#            free money meeting
emails = np.array([[1, 1, 0],   # spam
                   [1, 0, 0],   # spam
                   [0, 1, 0],   # spam
                   [0, 0, 1],   # ham
                   [1, 0, 1],   # ham
                   [0, 1, 1]])  # ham
labels = np.array([1, 1, 1, 0, 0, 0])
query = np.array([1, 0, 1])                                    # contains "free" and "meeting"

for alpha in (0.0, 1.0):
    prior = np.array([np.mean(labels == c) for c in (0, 1)])
    counts = np.array([emails[labels == c].sum(axis=0) for c in (0, 1)])              # n_cj
    n_c = np.array([(labels == c).sum() for c in (0, 1)])
    theta = (counts + alpha) / (n_c[:, None] + 2 * alpha)                              # P(word present | class)
    lik = np.where(query == 1, theta, 1 - theta).prod(axis=1)                          # P(query | class)
    joint = prior * lik
    print(f"alpha = {alpha}")
    print("  theta (rows: ham, spam):", theta.round(3).tolist())
    print("  P(c) P(x|c) (ham, spam):", joint.round(4).tolist(), "-> P(spam | x) =", (joint[1] / joint.sum()).round(3))

Without smoothing, "meeting" never appears in spam, so $P(\text{meeting}\mid \text{spam}) = 0$ and the e-mail can
**never** be spam, however much else it looks like it. **Laplace smoothing** adds $\alpha$ imaginary occurrences of
each outcome: $\theta_{cj} = (n_{cj} + \alpha)/(n_c + 2\alpha)$ for a binary feature. Now the probability is small
but positive.

### Bernoulli Naive Bayes on the spam data

Binary features: "the word (or character) appears at all", $b_j = [x_j > 0]$.

In [ ]:
spam = pl.read_csv(DATA / "spambase.csv.zst")
names = [c for c in spam.columns if c != "spam"]
X_raw = spam.select(names).to_numpy().astype(float)
y = spam["spam"].to_numpy().astype(int)
idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=0.2, stratify=y, random_state=0)
ytr, yte = y[idx_tr], y[idx_te]

B = (X_raw > 0).astype(int)
bnb = BernoulliNB(alpha=1.0).fit(B[idx_tr], ytr)
pred = bnb.predict(B[idx_te])
print("Bernoulli NB: accuracy", round(float(np.mean(pred == yte)), 3), "MCC", round(float(matthews_corrcoef(yte, pred)), 3))

# Which words say 'spam' most? The log-likelihood ratio log P(word present | spam) / P(word present | ham).
llr = bnb.feature_log_prob_[1] - bnb.feature_log_prob_[0]
order = np.argsort(llr)
print(pl.DataFrame({"feature": [names[i] for i in order[::-1][:6]], "log_ratio": llr[order[::-1][:6]].round(2)}))
print(pl.DataFrame({"feature": [names[i] for i in order[:6]], "log_ratio": llr[order[:6]].round(2)}))

# The same log-space score, computed by hand with jax.numpy, equals scikit-learn's.
theta = jnp.exp(jnp.asarray(bnb.feature_log_prob_))
joint = jnp.asarray(bnb.class_log_prior_) + jnp.where(B[idx_te][:, None, :] == 1, jnp.log(theta), jnp.log1p(-theta)[None]).sum(axis=-1)
print("max |joint log-probability, ours - sklearn| =", float(jnp.abs(joint - bnb.predict_joint_log_proba(B[idx_te])).max()))  # pyright: ignore[reportOperatorIssue]

**Smoothing matters when data is scarce.** Train on only the first $n$ e-mails, with $\alpha = 0$ (no smoothing;
scikit-learn refuses $\alpha = 0$, so we count by hand) and $\alpha = 1$: with no smoothing, any test e-mail with a word
unseen in one class gets a $-\infty$ score for that class.

In [ ]:
def bernoulli_scores(n, alpha):
    sub = idx_tr[:n]
    counts = np.array([B[sub][y[sub] == c].sum(axis=0) for c in (0, 1)])
    n_c = np.array([(y[sub] == c).sum() for c in (0, 1)])
    theta_ = (counts + alpha) / (n_c[:, None] + 2 * alpha)
    with np.errstate(divide="ignore"):
        per_feature = np.where(B[idx_te][:, None, :] == 1, np.log(theta_), np.log1p(-theta_))
        return np.log(n_c / n_c.sum()) + per_feature.sum(axis=-1)


for n in (20, 40, 100, 1000):
    for alpha in (0.0, 1.0):
        s = bernoulli_scores(n, alpha)
        print(f"n = {n:4d}, alpha = {alpha}: accuracy {np.mean(s.argmax(axis=1) == yte):.3f}, test e-mails with an impossible class {int(np.isneginf(s).any(axis=1).sum()):3d}")

## N2 — Continuous features: Gaussian Naive Bayes

For a real-valued feature, $P(x_j \mid c)$ is a density. **Gaussian NB** assumes
$x_j \mid c \sim \mathcal{N}(\mu_{cj}, \sigma_{cj}^2)$; the maximum-likelihood estimates are the class mean and the
class variance (divide by $n$). Take one feature: `capital_run_length_average` (the average length of runs of capital
letters, high in shouting spam), on a log scale.

In [ ]:
j = names.index("capital_run_length_average")
feature = np.log1p(X_raw[:, j])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
bins = np.linspace(feature.min(), feature.max(), 40)
grid = np.linspace(feature.min(), feature.max(), 300)
for c, color, label in ((0, "tab:blue", "ham"), (1, "tab:red", "spam")):
    vals = feature[idx_tr][ytr == c]
    mu, sd = vals.mean(), vals.std()
    axes[0].hist(vals, bins=bins, density=True, alpha=0.4, color=color, label=label)
    axes[0].plot(grid, np.exp(-0.5 * ((grid - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi)), color=color)
    print(f"{label}: mean {mu:.3f}, std {sd:.3f}")
axes[0].set(xlabel="log(1 + capital_run_length_average)", title="two Gaussians, one per class")
axes[0].legend()

raw_vals = X_raw[idx_tr, j]
axes[1].hist(raw_vals[raw_vals < 20], bins=60, color="tab:gray")
axes[1].set(xlabel="capital_run_length_average (raw, values < 20)", title="raw scale: far from Gaussian")
plt.show()

In [ ]:
X_log = np.log1p(X_raw)
for name, X_ in (("raw features", X_raw), ("log(1 + x)", X_log)):
    g = GaussianNB().fit(X_[idx_tr], ytr)
    p = g.predict(X_[idx_te])
    print(f"Gaussian NB on {name:<13}: accuracy {np.mean(p == yte):.3f}, MCC {matthews_corrcoef(yte, p):.3f}")

# Hand-made Gaussian NB: means, variances, log-densities summed over the features.
g = GaussianNB().fit(X_log[idx_tr], ytr)
mean = np.array([X_log[idx_tr][ytr == c].mean(axis=0) for c in (0, 1)])
var = np.array([X_log[idx_tr][ytr == c].var(axis=0) for c in (0, 1)]) + g.epsilon_
log_dens = -0.5 * (np.log(2 * np.pi * var)[None] + (X_log[idx_te][:, None, :] - mean[None]) ** 2 / var[None]).sum(axis=-1)
joint = np.log([np.mean(ytr == c) for c in (0, 1)]) + log_dens
print("max |joint log-probability, ours - sklearn| =", float(np.abs(joint - g.predict_joint_log_proba(X_log[idx_te])).max()))

**Discretizing instead.** Another way to handle a continuous feature: cut it into bins (here 4 quantile bins) and treat
the bin as a *category* (`CategoricalNB`, the multi-valued generalization of Bernoulli). No distribution is assumed.

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")                    # most word frequencies are 0 for most e-mails: their bins collapse
    kb = KBinsDiscretizer(n_bins=4, encode="ordinal", strategy="quantile").fit(X_raw[idx_tr])
    Z = np.asarray(kb.transform(X_raw)).astype(int)
cat = CategoricalNB(alpha=1.0, min_categories=4).fit(Z[idx_tr], ytr)
p = cat.predict(Z[idx_te])
print(f"Categorical NB, 4 quantile bins: accuracy {np.mean(p == yte):.3f}, MCC {matthews_corrcoef(yte, p):.3f}")

**Decision boundaries.** Gaussian NB with *different variances per class* has a **quadratic** boundary; logistic
regression a line. Two features of the spam data show the difference.

In [ ]:
cols = [names.index("capital_run_length_average"), names.index("word_freq_free")]
F = np.log1p(X_raw[:, cols])
gx, gy = np.meshgrid(np.linspace(F[:, 0].min(), F[:, 0].max(), 250), np.linspace(F[:, 1].min(), F[:, 1].max(), 250))
grid2 = np.c_[gx.ravel(), gy.ravel()]
sc2 = StandardScaler().fit(F[idx_tr])
models = {"Gaussian NB (quadratic)": GaussianNB().fit(F[idx_tr], ytr), "logistic (linear)": LogisticRegression().fit(sc2.transform(F[idx_tr]), ytr)}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, (title, model) in zip(axes, models.items()):
    scaled = "logistic" in title
    zz = model.predict(sc2.transform(grid2) if scaled else grid2).reshape(gx.shape)
    acc = model.score(sc2.transform(F[idx_te]) if scaled else F[idx_te], yte)
    ax.contourf(gx, gy, zz, levels=[-0.5, 0.5, 1.5], colors=["#cfe3f5", "#fbdcc4"])
    show = np.random.default_rng(0).choice(idx_te, 300, replace=False)
    ax.scatter(*F[show][y[show] == 0].T, s=8, color="tab:blue")
    ax.scatter(*F[show][y[show] == 1].T, s=8, color="tab:red")
    ax.set(title=f"{title}: accuracy {acc:.3f}", xlabel="log capital_run_length_average", ylabel="log word_freq_free")
plt.tight_layout()
plt.show()

## N3 — Naive Bayes versus logistic regression

Naive Bayes is **generative**: it models how each class produces the features, $P(x \mid c)$, then inverts with Bayes'
rule. Logistic regression is **discriminative**: it models $P(c \mid x)$ directly. Two consequences:

1. NB has few parameters to estimate and a strong assumption: it learns fast from little data but plateaus (its bias
   is the independence assumption).
2. When features are **correlated**, NB counts the same evidence several times: its probabilities are over-confident.

In [ ]:
scaler = StandardScaler().fit(X_log[idx_tr])
Xs_tr, Xs_te = scaler.transform(X_log[idx_tr]), scaler.transform(X_log[idx_te])
rng = np.random.default_rng(0)
sizes = [20, 50, 100, 300, 1000, len(idx_tr)]
curves = {"logistic": [], "Bernoulli NB": [], "Gaussian NB": []}
for n in sizes:
    acc = {k: [] for k in curves}
    for _ in range(10 if n < len(idx_tr) else 1):
        s = rng.choice(len(idx_tr), n, replace=False)
        if len(np.unique(ytr[s])) < 2:
            continue
        acc["logistic"].append(LogisticRegression(C=1.0, max_iter=5000).fit(Xs_tr[s], ytr[s]).score(Xs_te, yte))
        acc["Bernoulli NB"].append(BernoulliNB().fit(B[idx_tr][s], ytr[s]).score(B[idx_te], yte))
        acc["Gaussian NB"].append(GaussianNB().fit(X_log[idx_tr][s], ytr[s]).score(X_log[idx_te], yte))
    for k in curves:
        curves[k].append(float(np.mean(acc[k])))
for k, v in curves.items():
    plt.plot(sizes, v, "o-", label=k)
    print(f"{k:<13}", [round(x, 3) for x in v])
plt.xscale("log")
plt.xlabel("training examples (log)")
plt.ylabel("test accuracy")
plt.legend()
plt.show()

**Probabilities.** All three give $P(\text{spam}\mid x)$, but the numbers are not equally trustworthy. The
**log-loss** (cross-entropy) punishes confident mistakes: NB multiplies many correlated factors as if they were independent
evidence, so its probabilities are pushed to 0 or 1.

In [ ]:
lr = LogisticRegression(C=1.0, max_iter=5000).fit(Xs_tr, ytr)
gnb = GaussianNB().fit(X_log[idx_tr], ytr)
bnb = BernoulliNB().fit(B[idx_tr], ytr)
probas = {"logistic": lr.predict_proba(Xs_te), "Bernoulli NB": bnb.predict_proba(B[idx_te]), "Gaussian NB": gnb.predict_proba(X_log[idx_te])}
for name, p_ in probas.items():
    print(f"{name:<13} accuracy {np.mean(p_.argmax(axis=1) == yte):.3f}, log-loss {log_loss(yte, np.clip(p_, 1e-15, 1)):.3f}")

A **reliability table** compares the predicted probability with the observed spam fraction in each bin.

In [ ]:
edges = np.linspace(0, 1, 6)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (title, proba) in zip(axes, {"logistic": lr.predict_proba(Xs_te)[:, 1], "Gaussian NB": gnb.predict_proba(X_log[idx_te])[:, 1]}.items()):
    ax.hist(proba, bins=20, color="tab:gray")
    ax.set(title=f"{title}: predicted P(spam)", xlabel="probability")
    which = np.digitize(proba, edges[1:-1])
    rows = [(f"{edges[b]:.1f}-{edges[b + 1]:.1f}", int((which == b).sum()), float(yte[which == b].mean()) if (which == b).any() else float("nan")) for b in range(5)]
    print(title, [(r[0], r[1], round(r[2], 2)) for r in rows])
plt.tight_layout()
plt.show()

## N4 — Calibration

A probability is **calibrated** when it means what it says: among the e-mails given $P(\text{spam}) = 0.8$, about
80% are spam. Three ways to measure it:

* **Log-loss** (cross-entropy): $-\frac1n\sum_i \log p(y_i \mid x_i)$; punishes confident mistakes hard.
* **Brier score:** $\frac1n\sum_i (p_i - y_i)^2$, the mean squared error of the probability; bounded in $[0, 1]$.
* **Expected calibration error (ECE):** the gap between predicted and observed frequency, weighted over probability
  bins.

In [ ]:
def ece(p, y_true, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    which = np.digitize(p, edges[1:-1])
    return sum((which == b).mean() * abs(y_true[which == b].mean() - p[which == b].mean()) for b in range(bins) if (which == b).any())


def report(name, p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    print(f"{name:<28} accuracy {np.mean((p >= 0.5) == yte):.3f}  log-loss {log_loss(yte, p):.3f}  Brier {brier_score_loss(yte, p):.3f}  ECE {ece(p, yte):.3f}")


lr = LogisticRegression(C=1.0, max_iter=5000).fit(Xs_tr, ytr)
gnb = GaussianNB().fit(X_log[idx_tr], ytr)
bnb = BernoulliNB().fit(B[idx_tr], ytr)
report("logistic regression", lr.predict_proba(Xs_te)[:, 1])
report("Bernoulli NB", bnb.predict_proba(B[idx_te])[:, 1])
report("Gaussian NB", gnb.predict_proba(X_log[idx_te])[:, 1])

**Calibrating a classifier.** Keep the classifier, and learn a monotone map from its score to a probability on held-out
data:

* **Platt scaling** (`method="sigmoid"`): fit a logistic regression on the score, $p = \sigma(a s + b)$, two
  parameters.
* **Isotonic regression** (`method="isotonic"`): any non-decreasing step function; more flexible, needs more data.

`CalibratedClassifierCV` does it inside cross-validation (the calibration data are never the training data).

In [ ]:
report("Gaussian NB", gnb.predict_proba(X_log[idx_te])[:, 1])
for method in ("sigmoid", "isotonic"):
    cg = CalibratedClassifierCV(GaussianNB(), method=method, cv=5).fit(X_log[idx_tr], ytr)
    report(f"Gaussian NB + {method}", cg.predict_proba(X_log[idx_te])[:, 1])
report("Bernoulli NB", bnb.predict_proba(B[idx_te])[:, 1])
for method in ("sigmoid", "isotonic"):
    cb = CalibratedClassifierCV(BernoulliNB(), method=method, cv=5).fit(B[idx_tr], ytr)
    report(f"Bernoulli NB + {method}", cb.predict_proba(B[idx_te])[:, 1])

In [ ]:
cg_iso = CalibratedClassifierCV(GaussianNB(), method="isotonic", cv=5).fit(X_log[idx_tr], ytr)
curves = {"logistic": lr.predict_proba(Xs_te)[:, 1], "Gaussian NB": gnb.predict_proba(X_log[idx_te])[:, 1], "Gaussian NB + isotonic": cg_iso.predict_proba(X_log[idx_te])[:, 1]}
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6), sharey=True)
edges = np.linspace(0, 1, 11)
for ax, (title, proba) in zip(axes, curves.items()):
    which = np.digitize(proba, edges[1:-1])
    mid = [proba[which == b].mean() for b in range(10) if (which == b).sum() >= 5]
    obs = [yte[which == b].mean() for b in range(10) if (which == b).sum() >= 5]
    ax.plot([0, 1], [0, 1], "k--")
    ax.plot(mid, obs, "o-")
    ax.set(title=title, xlabel="predicted probability")
axes[0].set_ylabel("observed spam fraction")
plt.tight_layout()
plt.show()

Two things to notice. First, Gaussian NB is the worst calibrated and the calibration repairs most of it (its log-loss
falls from about 3.1 to about 0.2). Second, the *accuracy* of the isotonic version also changes: the calibrated
classifier is an ensemble of five models fitted on different folds, not a pure re-labelling of one model.
A calibration map that is **monotone** never changes the ranking of the e-mails, only the probabilities.

## N5 — Decisions with costs

Given a calibrated $P(\text{spam}\mid x)$, the best decision minimizes the **expected cost**. Let a false positive
(a real e-mail hidden) cost $c_{FP}$ and a false negative (a spam delivered) cost $c_{FN}$. Flag the e-mail as spam
when

$$p\,c_{FN} \;>\; (1-p)\,c_{FP} \quad\Longleftrightarrow\quad p > \frac{c_{FP}}{c_{FP} + c_{FN}} = t^\star$$

Equal costs give the familiar 0.5; if a hidden e-mail costs ten times a delivered spam, $t^\star = 10/11 = 0.909$.

In [ ]:
p_lr = lr.predict_proba(Xs_te)[:, 1]


def cost(p, threshold, c_fp, c_fn):
    tn, fp, fn, tp = confusion_matrix(yte, (p >= threshold).astype(int)).ravel()
    return (c_fp * fp + c_fn * fn) / len(yte), int(fp), int(fn)


grid = np.linspace(0.02, 0.99, 98)
for c_fp, c_fn in ((1, 1), (10, 1), (1, 5)):
    t_star = c_fp / (c_fp + c_fn)
    at_half = cost(p_lr, 0.5, c_fp, c_fn)
    at_star = cost(p_lr, t_star, c_fp, c_fn)
    scan = [cost(p_lr, t, c_fp, c_fn)[0] for t in grid]
    print(f"costs FP:FN = {c_fp}:{c_fn}  t* = {t_star:.3f} | threshold 0.5: cost {at_half[0]:.3f} (FP {at_half[1]}, FN {at_half[2]}) | "
          f"threshold t*: cost {at_star[0]:.3f} (FP {at_star[1]}, FN {at_star[2]}) | best on the test set: t = {grid[int(np.argmin(scan))]:.2f}, cost {min(scan):.3f}")

c_fp, c_fn = 10, 1
plt.plot(grid, [cost(p_lr, t, c_fp, c_fn)[0] for t in grid])
plt.axvline(c_fp / (c_fp + c_fn), color="tab:red", ls="--", label=r"Bayes threshold $t^\star = 0.909$")
plt.axvline(0.5, color="gray", ls=":", label="default 0.5")
plt.xlabel("threshold")
plt.ylabel("expected cost per e-mail (FP : FN = 10 : 1)")
plt.legend()
plt.show()

The theoretical threshold is close to the best one found by scanning the test set (which we must not do in practice),
*if the probabilities are calibrated*. With an over-confident model the threshold computed from costs would be wrong:
**calibrate first, then decide**.

## Summary

* **Naive Bayes** = Bayes' rule + conditional independence; training is counting (Bernoulli, categorical, with Laplace
  smoothing = a Beta prior) or means and variances (Gaussian).
* Word frequencies are not Gaussian: the coarsest discrete representation wins among the NB models; the logistic
  regression wins overall, most clearly with more data.
* Naive Bayes is a good classifier but a poor probability estimator; Platt or isotonic scaling repairs the
  probabilities.
* With calibrated probabilities the best decision follows from the costs: $t^\star = c_{FP}/(c_{FP} + c_{FN})$.